In [1]:
import sys; print(sys.executable)

/home/linux1/tp/bin/python3


In [2]:
import platform, os, sys, importlib, subprocess
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print(platform.machine(), "|", platform.platform(), "|", sys.version.split()[0])
print("CPUs:", os.cpu_count())
print(sh("free -h")); print(sh("df -h /"))
print(sh("grep -m1 '^features' /proc/cpuinfo"))   # look for 'nnpa' before any Telum/NNPA claim
for m in ["numpy","pandas","sklearn","xgboost","networkx","matplotlib","pyarrow"]:
    try:
        mod = importlib.import_module(m); print(f"{m:11} {getattr(mod,'__version__','?')}")
    except Exception as e:
        print(f"{m:11} MISSING ({type(e).__name__})")

s390x | Linux-6.8.0-147-generic-s390x-with-glibc2.39 | 3.12.3
CPUs: 3
               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       1.3Gi       6.4Gi        18Mi       234Mi       6.5Gi
Swap:          4.0Gi          0B       4.0Gi

Filesystem      Size  Used Avail Use% Mounted on
/dev/dasda1      50G  9.6G   38G  21% /

features	: esan3 zarch stfle msa ldisp eimm dfp edat etf3eh highgprs te vx vxd vxe gs vxe2 vxp sort dflt vxp2 nnpa sie 

numpy       1.26.4
pandas      2.1.4
sklearn     1.4.1.post1
xgboost     MISSING (ModuleNotFoundError)
networkx    2.8.8
matplotlib  3.6.3
pyarrow     MISSING (ModuleNotFoundError)


In [3]:
pip install --prefer-binary numpy pandas scikit-learn networkx xgboost pyarrow

  Using cached xgboost-3.4.1.tar.gz (1.2 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyarrow-26.0.0.tar.gz (1.2 MB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × installing build dependencies for pyarrow did not run successfully.
  │ exit code: 1
  ╰─> [493 lines of output]
      Ignoring libcst: markers 'python_version >= "3.15"' don't match your environment
        Using cached scikit_build_core-1.1.1-py3-none-any.whl.metadata (19 kB)
        Using cached cython-3.3.0-py3-none-any.whl.metadata (28 kB)
        Using cached libcst-1.9.0.tar.gz (914 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
       

In [4]:
import platform, os, sys, importlib, subprocess
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print(sys.executable)
print(platform.machine(), "|", platform.platform(), "|", sys.version.split()[0])
print("CPUs:", os.cpu_count())
print(sh("free -h")); print(sh("df -h /"))
print(sh("grep -m1 '^features' /proc/cpuinfo"))
for m in ["numpy","pandas","scipy","sklearn","xgboost","networkx","matplotlib"]:
    try:
        mod = importlib.import_module(m); print(f"{m:11} {getattr(mod,'__version__','?'):10} {mod.__file__}")
    except Exception as e:
        print(f"{m:11} MISSING ({type(e).__name__}: {e})")

/home/linux1/tp/bin/python3
s390x | Linux-6.8.0-147-generic-s390x-with-glibc2.39 | 3.12.3
CPUs: 3
               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       1.4Gi       6.2Gi        18Mi       375Mi       6.4Gi
Swap:          4.0Gi          0B       4.0Gi

Filesystem      Size  Used Avail Use% Mounted on
/dev/dasda1      50G  9.6G   38G  21% /

features	: esan3 zarch stfle msa ldisp eimm dfp edat etf3eh highgprs te vx vxd vxe gs vxe2 vxp sort dflt vxp2 nnpa sie 

numpy       1.26.4     /usr/lib/python3/dist-packages/numpy/__init__.py
pandas      2.1.4      /usr/lib/python3/dist-packages/pandas/__init__.py
scipy       1.11.4     /usr/lib/python3/dist-packages/scipy/__init__.py
sklearn     1.4.1.post1 /usr/lib/python3/dist-packages/sklearn/__init__.py
xgboost     MISSING (ModuleNotFoundError: No module named 'xgboost')
networkx    2.8.8      /usr/lib/python3/dist-packages/networkx/__init__.py
matplotlib  3.6.3      /usr/lib/python3/dist-pa

In [6]:
import glob, time, platform, numpy as np, pandas as pd

path = glob.glob("/home/linux1/trustpulse/data/*.csv")[0]
t0 = time.time()
# Balance columns are excluded: PaySim cancels fraud transactions, so balances leak the label.
# isFlaggedFraud is the simulator's own rule output, not something a bank sees pre-payment.
df = pd.read_csv(path,
                 usecols=["step", "type", "amount", "nameOrig", "nameDest", "isFraud"],
                 dtype={"step": "int16", "type": "category", "amount": "float64",
                        "nameOrig": "category", "nameDest": "category", "isFraud": "int8"})
df["orig_id"] = df.nameOrig.cat.codes.astype("int32")
df["dest_id"] = df.nameDest.cat.codes.astype("int32")
df["dest_is_merchant"] = df.nameDest.str.startswith("M").astype("int8")
n_orig_unique = df.orig_id.nunique()
df = df.drop(columns=["nameOrig", "nameDest"])

print(f"loaded {len(df):,} rows in {time.time()-t0:.1f}s | steps {df.step.min()}-{df.step.max()}")
print(f"fraud rate {df.isFraud.mean():.5f} | unique senders {n_orig_unique:,}")
print(df.groupby("type", observed=True).isFraud.agg(["size", "sum"]))

ValueError: Usecols do not match columns, columns expected but not found: ['type', 'amount', 'isFraud', 'nameOrig', 'step', 'nameDest']

In [ ]:
df["log_amount"] = np.log1p(df.amount)
df["hour"] = (df.step % 24).astype("int8")
df["type_code"] = df.type.cat.codes.astype("int8")

# Per-destination history, aggregated per step, then shifted so a row only sees prior steps.
g = df.groupby(["dest_id", "step"]).agg(n=("amount", "size"), amt=("amount", "sum")).reset_index()
gg = g.groupby("dest_id")
g["dest_prev_n"] = gg["n"].cumsum() - g["n"]
g["dest_prev_amt"] = gg["amt"].cumsum() - g["amt"]
g["dest_last_step"] = gg["step"].shift(1)
df = df.merge(g[["dest_id", "step", "dest_prev_n", "dest_prev_amt", "dest_last_step"]],
              on=["dest_id", "step"], how="left")
del g, gg

df["dest_prev_mean"] = (df.dest_prev_amt / df.dest_prev_n).fillna(0)
df["dest_gap"] = (df.step - df.dest_last_step).fillna(-1)
df["dest_is_new"] = (df.dest_prev_n == 0).astype("int8")
df["amt_vs_dest_mean"] = df.amount / (df.dest_prev_mean + 1)

FEATURES = ["log_amount", "hour", "type_code", "dest_is_merchant", "dest_prev_n",
            "dest_prev_amt", "dest_prev_mean", "dest_gap", "dest_is_new", "amt_vs_dest_mean"]
df[FEATURES].describe().T

In [ ]:
TRAIN_END, VAL_END = 400, 550
tr = df[df.step <= TRAIN_END]
va = df[(df.step > TRAIN_END) & (df.step <= VAL_END)]
te = df[df.step > VAL_END]
for name, d in [("train", tr), ("val", va), ("test", te)]:
    print(f"{name:5} rows {len(d):>9,} fraud {d.isFraud.sum():>5} rate {d.isFraud.mean():.4f}")

# Downsample legit rows in train for speed on 3 vCPUs; ranking metrics are unaffected.
rng = np.random.default_rng(42)
trs = tr[(tr.isFraud == 1) | (rng.random(len(tr)) < 0.2)]
print("train sample", len(trs), "fraud", trs.isFraud.sum())

In [ ]:
try:
    import xgboost as xgb
    spw = (trs.isFraud == 0).sum() / trs.isFraud.sum()
    model = xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1,
                              tree_method="hist", n_jobs=3, scale_pos_weight=spw,
                              eval_metric="aucpr", random_state=42)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=63,
                                           early_stopping=False, class_weight="balanced",
                                           random_state=42)
t0 = time.time()
model.fit(trs[FEATURES], trs.isFraud)
print(type(model).__name__, f"trained in {time.time()-t0:.1f}s")

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

def evaluate(d, name, budget=0.005):
    p = model.predict_proba(d[FEATURES])[:, 1]
    y = d.isFraud.values
    top = np.argsort(-p)[:max(1, int(budget * len(d)))]
    print(f"{name}: PR-AUC {average_precision_score(y, p):.4f} | ROC-AUC {roc_auc_score(y, p):.4f} | "
          f"recall@{budget:.1%} alerts {y[top].sum()/y.sum():.3f} | precision {y[top].mean():.3f}")
    return p

p_va = evaluate(va, "val")
p_te = evaluate(te, "test")

x1 = te[FEATURES].iloc[:1]
t = time.perf_counter()
for _ in range(200):
    model.predict_proba(x1)
single_ms = (time.perf_counter() - t) / 200 * 1000
xb = te[FEATURES].iloc[:10000]
t = time.perf_counter(); model.predict_proba(xb)
batch_us = (time.perf_counter() - t) / 10000 * 1e6
print(f"latency on {platform.machine()}: single txn {single_ms:.2f} ms | batch {batch_us:.1f} µs/txn")

In [ ]:
import joblib, os
os.makedirs("/home/linux1/trustpulse/artifacts", exist_ok=True)
joblib.dump(model, "/home/linux1/trustpulse/artifacts/baseline.joblib")
df.to_pickle("/home/linux1/trustpulse/artifacts/paysim_features.pkl")

## Part 2: Network-aware engine on IBM AML (HI-Small)

In [ ]:
import time, numpy as np, pandas as pd
DATA = "/home/linux1/trustpulse/data"
COLS = ["ts", "from_bank", "from_acct", "to_bank", "to_acct", "amt_recv", "cur_recv",
        "amt_paid", "cur_paid", "fmt", "is_laundering"]
DTYPES = {"from_bank": str, "from_acct": str, "to_bank": str, "to_acct": str,
          "cur_recv": "category", "cur_paid": "category", "fmt": "category", "is_laundering": "int8"}

t0 = time.time()
aml = pd.read_csv(f"{DATA}/HI-Small_Trans.csv", header=0, names=COLS, dtype=DTYPES)
aml["ts"] = pd.to_datetime(aml.ts, format="%Y/%m/%d %H:%M")

# One shared integer id per (bank, account) so senders and receivers live in the same graph.
src = aml.from_bank + "_" + aml.from_acct
dst = aml.to_bank + "_" + aml.to_acct
codes, uniques = pd.factorize(pd.concat([src, dst], ignore_index=True))
aml["src"] = codes[:len(aml)].astype("int32")
aml["dst"] = codes[len(aml):].astype("int32")
del src, dst, codes

aml = aml.sort_values("ts", kind="stable").reset_index(drop=True)
aml["txn_id"] = np.arange(len(aml), dtype="int32")

print(f"{len(aml):,} rows in {time.time()-t0:.1f}s | {aml.ts.min()} to {aml.ts.max()}")
print(f"accounts {len(uniques):,} | laundering rate {aml.is_laundering.mean():.5f} | "
      f"self-transfers {(aml.src == aml.dst).mean():.3f}")
print(aml.groupby("fmt", observed=True).is_laundering.agg(["size", "sum"]))

In [ ]:
rows, pid, ptype = [], -1, None
with open(f"{DATA}/HI-Small_Patterns.txt") as f:
    for line in f:
        line = line.strip()
        if line.startswith("BEGIN"):
            pid += 1
            ptype = line.split("-", 1)[1].split(":")[0].strip()
        elif line and not line.startswith("END"):
            rows.append(line.split(",") + [pid, ptype])

pat = pd.DataFrame(rows, columns=COLS + ["pattern_id", "pattern_type"])
pat["ts"] = pd.to_datetime(pat.ts, format="%Y/%m/%d %H:%M")
pat["amt_paid"] = pat.amt_paid.astype(float)
KEY = ["ts", "from_bank", "from_acct", "to_bank", "to_acct", "amt_paid"]
pat = pat.drop_duplicates(KEY)[KEY + ["pattern_id", "pattern_type"]]

aml = aml.merge(pat, on=KEY, how="left")
aml["pattern_id"] = aml.pattern_id.fillna(-1).astype("int32")

lab = aml.is_laundering == 1
print(f"patterns {pid+1:,} | laundering txns in a pattern: {(lab & (aml.pattern_id >= 0)).sum():,} "
      f"of {lab.sum():,} | pattern txns not labelled: {(~lab & (aml.pattern_id >= 0)).sum():,}")

ps = aml[aml.pattern_id >= 0].groupby("pattern_id").agg(
    ptype=("pattern_type", "first"), n=("txn_id", "size"), start=("ts", "min"), end=("ts", "max"))
ps["hours"] = (ps.end - ps.start).dt.total_seconds() / 3600
print(ps.groupby("ptype").agg(patterns=("n", "size"), txns_median=("n", "median"),
                              hours_median=("hours", "median"), hours_p90=("hours", lambda h: h.quantile(0.9))))

In [ ]:
daily = aml.groupby(aml.ts.dt.date).agg(txns=("txn_id", "size"), laundering=("is_laundering", "sum"))
daily["rate"] = daily.laundering / daily.txns
print(daily)
aml = aml.drop(columns=["from_acct", "to_acct"])
aml.to_pickle("/home/linux1/trustpulse/artifacts/aml_hi_small.pkl")

In [ ]:
aml = aml[aml.ts < "2022-09-11"].reset_index(drop=True)
aml["txn_id"] = np.arange(len(aml), dtype="int32")

# Exchange rates implied by the data itself: amount received in USD / amount paid in currency X.
fx = aml[(aml.cur_recv == "US Dollar") & (aml.cur_paid != "US Dollar")]
rate = (fx.amt_recv / fx.amt_paid).groupby(fx.cur_paid, observed=True).median()
rate["US Dollar"] = 1.0
print(rate.round(4))
aml["amt_usd"] = aml.cur_paid.map(rate).astype(float).fillna(1.0) * aml.amt_paid
aml["cross_currency"] = (aml.cur_recv.astype(str) != aml.cur_paid.astype(str)).astype("int8")
del fx

In [ ]:
t0 = time.time()
n = len(aml)
first_pair = ~aml.duplicated(["src", "dst"])          # first time this sender pays this receiver
aml["pair_prev_n"] = aml.groupby(["src", "dst"]).cumcount().astype("int32")

# Each transaction becomes two events: an outgoing event for the sender, an incoming one for the receiver.
ev = pd.DataFrame({
    "acct": np.concatenate([aml.src.values, aml.dst.values]),
    "txn": np.concatenate([aml.txn_id.values, aml.txn_id.values]),
    "is_in": np.concatenate([np.zeros(n, "int8"), np.ones(n, "int8")]),
    "amt": np.concatenate([aml.amt_usd.values, aml.amt_usd.values]),
    "first": np.concatenate([first_pair.values, first_pair.values]).astype("int8"),
    "ts": np.concatenate([aml.ts.values, aml.ts.values]),
}).sort_values(["acct", "txn", "is_in"], kind="stable").reset_index(drop=True)

# Every running total subtracts the current event, so a row only sees strictly earlier activity.
def excl_cumsum(x):
    return x.groupby(ev.acct, sort=False).cumsum() - x

is_out = 1 - ev.is_in
ev["in_n"], ev["out_n"] = excl_cumsum(ev.is_in), excl_cumsum(is_out)
ev["in_amt"], ev["out_amt"] = excl_cumsum(ev.amt * ev.is_in), excl_cumsum(ev.amt * is_out)
ev["in_uniq"], ev["out_uniq"] = excl_cumsum(ev["first"] * ev.is_in), excl_cumsum(ev["first"] * is_out)
g = ev.groupby("acct", sort=False).ts
ev["age_h"] = (ev.ts - g.transform("min")).dt.total_seconds() / 3600
ev["gap_h"] = ((ev.ts - g.shift(1)).dt.total_seconds() / 3600).fillna(-1)

ACCT = ["in_n", "out_n", "in_amt", "out_amt", "in_uniq", "out_uniq", "age_h", "gap_h"]
aml = (aml.join(ev[ev.is_in == 0].set_index("txn")[ACCT].add_prefix("src_"), on="txn_id")
          .join(ev[ev.is_in == 1].set_index("txn")[ACCT].add_prefix("dst_"), on="txn_id"))
del ev, g, is_out, first_pair

aml["log_amt"] = np.log1p(aml.amt_usd)
aml["hour"] = aml.ts.dt.hour.astype("int8")
aml["fmt_code"] = aml.fmt.cat.codes.astype("int8")
aml["cur_code"] = aml.cur_paid.cat.codes.astype("int8")
aml["self_txn"] = (aml.src == aml.dst).astype("int8")
aml["new_counterparty"] = (aml.pair_prev_n == 0).astype("int8")
# Mule signature: money that arrived is being pushed straight back out.
aml["src_passthrough"] = aml.src_out_amt / (aml.src_in_amt + 1)
aml["dst_passthrough"] = aml.dst_out_amt / (aml.dst_in_amt + 1)
aml["amt_vs_src_avg"] = aml.amt_usd / (aml.src_out_amt / aml.src_out_n.clip(lower=1) + 1)

LOCAL = ["log_amt", "hour", "fmt_code", "cur_code", "cross_currency", "self_txn"]
NET = (["pair_prev_n", "new_counterparty", "src_passthrough", "dst_passthrough", "amt_vs_src_avg"]
       + [f"{s}_{a}" for s in ("src", "dst") for a in ACCT])
print(f"features built in {time.time()-t0:.1f}s | {len(LOCAL)} local + {len(NET)} network")